## Sentiment Wikipedia-Artikel

Hinweis: Dieses Notebook wurde mit Gemini entwickelt um die Grundlagen der Sentiment-Analyse (als NLP-Methode) zu lernen. Es wurde zwischen NLTK und Spacy als Bibliothek abgewogen.

### Setup

In [1]:
import pandas as pd
import json
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize


In [2]:
nltk.download('punkt')
nltk.download('vader_lexicon')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Annette\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Annette\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
with open("../data/wikipedia_articles.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

sia = SentimentIntensityAnalyzer()
for article in articles[:2]:  # Teste erstmal nur die ersten zwei
    title = article.get("wikipedia_title")
    text = article.get("text_en", "")
    score = sia.polarity_scores(text)['compound']
    print(f"{title}: Score = {score}")

FC Bayern Munich: Score = 1.0
Borussia Dortmund: Score = 0.9999


Sentiment-Werte liegen zwischen -1 und 1. Beide Werte also sehr hoch und die Einschätzung sehr positiv. 

Abhilfe für Aussagekräftigere Sentiment-Analysen: Den Durchschnitt pro Satz berechnen d.h. Jeder Satz erhält eine eigene Sentiment-Analyse und dann errechnet man den Durchschnitt 

In [4]:
import json
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer

def test_wikipedia_sentiment_dataframe(file_path="../Data/wikipedia_articles.json"):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            articles = json.load(f)
    except FileNotFoundError:
        print(f"Fehler: Datei unter '{file_path}' nicht gefunden!")
        return pd.DataFrame()
        
    sia = SentimentIntensityAnalyzer()
    data = []
    
    for article in articles:
        verein = article.get("wikipedia_title")
        text = article.get("text_en", "")
        
        if text:
            # TRICK: Wir teilen den Text einfach am Punkt gefolgt von einem Leerzeichen.
            # Das ersetzt nltk.sent_tokenize perfekt und braucht keinen Download!
            saetze = text.split(". ")
            
            # Score für jeden einzelnen Satz berechnen
            satz_scores = [sia.polarity_scores(s)['compound'] for s in saetze if s.strip()]
            
            if satz_scores:
                avg_score = sum(satz_scores) / len(satz_scores)
                
                if avg_score >= 0.45:  
                    stimmung = "🟢 Sehr Positiv"
                elif avg_score >= 0.35:
                    stimmung = "🟠 Positiv"
                else:
                    stimmung = "🟡 Etwas durchwachsen"
                    
                data.append({
                    "Verein": verein,
                    "Ø Sentiment-Score": round(avg_score, 4),
                    "Grundstimmung": stimmung
                })
            
    df = pd.DataFrame(data)
    if not df.empty:
        df = df.sort_values(by="Ø Sentiment-Score", ascending=False)
    return df

# --- AUSFÜHRUNG ---
df_ergebnis = test_wikipedia_sentiment_dataframe()
df_ergebnis

,Verein,Ø Sentiment-Score,Grundstimmung
92,Le Havre AC,0.5625,🟢 Sehr Positiv
85,OGC Nice,0.3971,🟠 Positiv
5,Eintracht Frankfurt,0.3249,🟡 Etwas durchwachsen
59,Inter Milan,0.3163,🟡 Etwas durchwachsen
60,Juventus FC,0.3134,🟡 Etwas durchwachsen
...,...,...,...
68,Hellas Verona FC,0.0693,🟡 Etwas durchwachsen
49,Rayo Vallecano,0.0631,🟡 Etwas durchwachsen
52,Almería,0.0566,🟡 Etwas durchwachsen
57,Mallorca,0.0544,🟡 Etwas durchwachsen
